## Definición y análisis de métricas de evaluación

Nuestro dataset presenta una proporción mayoritaria de tweets neutrales, aproximadamente un 66,6%.

### 1. Métricas disponibles

| Métrica | ¿Qué nos permite analizar? |
|---|---|
| **Accuracy** | Qué porcentaje de tweets clasificamos correctamente. |
| **Precision por clase** | Qué tan confiable es una predicción positiva, negativa o neutral. |
| **Recall por clase** | Qué capacidad tiene el modelo para reconocer cada sentimiento. |
| **F1-score por clase** | El equilibrio entre detectar los ejemplos de una clase y evitar asignarla incorrectamente. |
| **Macro Precision** | La confiabilidad general de las predicciones sin favorecer clases frecuentes. |
| **Macro Recall** | La capacidad de reconocer cada sentimiento independientemente de su frecuencia. |
| **Macro F1** | El equilibrio entre Precision y Recall, dando igual importancia a cada sentimiento. |
| **Weighted F1** | El rendimiento general considerando la frecuencia de los sentimientos. |
| **Matriz de confusión** | Qué sentimientos confunde el modelo y con qué frecuencia. |

### 2. Consideraciones sobre el desbalance de clases

El predominio de tweets neutrales puede influir en el entrenamiento y la evaluación.

Por ejemplo, un clasificador que siempre prediga la clase neutral obtendría aproximadamente un **66,6% de Accuracy**, aunque no sea capaz de identificar tweets positivos ni negativos.

Este ejemplo demuestra que un Accuracy relativamente alto no necesariamente implica un modelo útil para todos los objetivos de clasificación.

Sin embargo, la frecuencia de la clase neutral también es relevante: si el objetivo fuera maximizar la cantidad total de tweets correctamente clasificados, su buen reconocimiento tendría una importancia considerable.

Por lo tanto, no debemos asumir que todas las clases tienen necesariamente la misma prioridad ni que las clases minoritarias deben recibir siempre mayor importancia.

### 3. Análisis de los errores de clasificación

Es necesario identificar qué consecuencias podrían tener las distintas equivocaciones del modelo.

| Sentimiento real | Predicción | Posible consecuencia |
|---|---|---|
| Negativo | Neutral | Se pierde una opinión negativa existente. |
| Positivo | Neutral | Se pierde una opinión positiva existente. |
| Neutral | Negativo | Se interpreta como crítica una publicación que no expresa una opinión negativa. |
| Neutral | Positivo | Se interpreta como favorable una publicación que no expresa una opinión positiva. |
| Negativo | Positivo | Se interpreta una opinión negativa como favorable. |
| Positivo | Negativo | Se interpreta una opinión positiva como desfavorable. |

La importancia relativa de estos errores dependerá del uso previsto para las predicciones.

En consecuencia, antes de seleccionar la métrica principal será necesario definir si queremos priorizar:

- La cantidad total de tweets correctamente clasificados.
- La detección de las opiniones positivas y negativas.
- La confiabilidad de las predicciones neutrales.
- El reconocimiento equilibrado de los tres sentimientos.
- La reducción de determinadas confusiones entre clases.

### 4. Criterio de selección

La métrica principal se establecerá en función de los objetivos del problema y del tipo de error que resulte más perjudicial.

Adicionalmente, utilizaremos métricas complementarias y la matriz de confusión para analizar el comportamiento de cada clasificador.

La métrica seleccionada se utilizará como criterio de optimización en `GridSearchCV`, permitiendo comparar los modelos bajo un procedimiento de validación cruzada consistente.

## ¿qué queremos optimizar?

### Escenario A: clasificar correctamente la mayor cantidad posible de tweets

Si nuestra prioridad es minimizar la cantidad total de clasificaciones incorrectas, independientemente del sentimiento, Accuracy es una elección directa.
Tiene sentido dar más peso a los neutrales porque representan la mayor parte de los tweets que esperamos encontrar, siempre que esa distribución sea representativa de los datos futuros.

### Escenario B: detectar las opiniones que expresan un sentimiento

Imaginá que la organización está especialmente interesada en identificar reacciones positivas y negativas frente a la IA.
En ese caso, clasificar esas opiniones como neutrales podría ser particularmente perjudicial.
Nos interesaría prestar mucha atención al Recall de P y N para no perder opiniones que sí expresan sentimiento.

### Escenario C: representar correctamente los tres sentimientos

Si la organización pretende estudiar la distribución de opiniones positivas, negativas y neutrales, podríamos argumentar que un modelo no debe sacrificar el reconocimiento de una categoría para favorecer otra.
En este escenario tienen sentido Macro Recall y Macro F1, pero por motivos diferentes.
Macro Recall prioriza la capacidad de identificar los ejemplos reales de cada clase. Macro F1 agrega la necesidad de equilibrar esa detección con la confiabilidad de las predicciones.
Ahí sí tendríamos una justificación para considerar F1, en lugar de elegirlo simplemente porque hay tres clases.

### Escenario D: priorizar la clasificación de los neutrales

Supongamos que queremos evitar atribuir sentimientos positivos o negativos a tweets que realmente son neutrales.
En ese caso, nos interesaría un Recall alto para NEU.
Pero si lo prioritario fuera evitar que opiniones reales queden etiquetadas como neutrales, buscaríamos una Precision alta para NEU.
Son objetivos diferentes, aunque ambos estén relacionados con la misma clase.

Viendo que el contexto es que, la organizacion quiere ver cual es la tendencia de la opinon de la ia, creo que lo mejor seria que el modelo tenga la capacidad de generalizar bien para cualquiera de las 3 clases.

La organización busca analizar la tendencia de la opinión pública sobre la inteligencia artificial. Para que ese análisis sea representativo, necesitamos que el clasificador pueda reconocer adecuadamente los tres sentimientos: positivo, negativo y neutral.

### Justificacion para la defensa del tp, no para meter en el notebook

El objetivo de la organización es analizar las tendencias de opinión del público hispanohablante respecto de la inteligencia artificial, mediante la clasificación automática de tweets en tres categorías: positivo, negativo y neutral.

Dado que buscamos identificar adecuadamente las distintas percepciones, consideramos importante que el modelo tenga un desempeño equilibrado en las tres clases, independientemente de su frecuencia en el dataset.

Esta decisión adquiere especial relevancia debido al desbalance observado, donde aproximadamente el 66,6% de los tweets pertenecen a la clase neutral. Si utilizáramos únicamente Accuracy, podríamos favorecer modelos con un buen rendimiento general debido a su capacidad para reconocer la clase mayoritaria, aunque presenten dificultades para identificar opiniones positivas o negativas.

Por este motivo, seleccionamos **Macro F1 como métrica principal para la optimización y comparación de modelos**.

Esta métrica combina Precision y Recall de cada clase mediante su media armónica y luego calcula el promedio de los F1 individuales, asignando la misma importancia a los tres sentimientos.

Consideramos que este criterio se alinea con el objetivo del problema, ya que buscamos tanto identificar las opiniones correspondientes a cada sentimiento como evitar asignar incorrectamente esas etiquetas a otros tweets.

No obstante, Macro F1 no describe por sí sola todos los errores del clasificador. Por eso complementaremos su análisis con:

- **Precision, Recall y F1-score por clase**, para identificar fortalezas y debilidades en el reconocimiento de cada sentimiento.
- **Accuracy**, para conocer la proporción global de predicciones correctas.
- **Matriz de confusión**, para analizar específicamente qué sentimientos tiende a confundir el modelo.
- **Comparación de la distribución real y predicha de sentimientos**, para estudiar posibles distorsiones en la tendencia general estimada.

De esta manera, la selección del modelo no dependerá únicamente de una puntuación global, sino también de su capacidad para ofrecer predicciones equilibradas y útiles para el objetivo de la organización.